In [12]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr
import io

# 假设你的CSV数据读取为 df
# df = pd.read_csv('your_data.csv')

# 示例数据加载 (忽略后面的空列)
# csv_data = """Environment,Sample_ID,Time,DS,DS_TPM,ADS,ADS_TPM
# spring,SRR21619491,2026/12/19,Cas,693303,Anti_CRISPR,26543
# spring,SRR21619492,2026/3/18,Cas,629643,Anti_CRISPR,34967.5
# spring,SRR21619493,2026/9/17,Cas,618192,Anti_CRISPR,27909.5
# spring,SRR21619494,2026/1/16,Cas,638587,Anti_CRISPR,14536.2
# AS,SRR30303755,201808,Cas,107816.3319,Anti_CRISPR,8242.994964
# AS,SRR30303793,201307,Cas,78518.42435,Anti_CRISPR,5483.380894
# """ # 这里为了简明仅截取部分做演示，实际请直接读取你的文件
df = pd.read_csv(r"D:\softerware_scientific_research\python_all_inone\py learning\pythonProject\prophage_project\huiyuan_ADS_DS\time_scale.csv", sep=',', header=0)

df.columns = df.columns.str.strip()
# ---------------- 1. 数据预处理与时间格式标准化 ----------------
def parse_date(date_str):
    """处理混合格式的时间字符串：YYYY/MM/DD 或 YYYYMM"""
    date_str = str(date_str).strip()
    try:
        if '/' in date_str:
            return pd.to_datetime(date_str, format='%Y/%m/%d')
        elif len(date_str) == 6: # 处理 201808 这种格式
            return pd.to_datetime(date_str, format='%Y%m')
        else:
            return pd.to_datetime(date_str)
    except Exception as e:
        return pd.NaT

df['Time_Parsed'] = df['Time'].apply(parse_date)

# 确保数据属于数值型
df['DS_TPM'] = pd.to_numeric(df['DS_TPM'], errors='coerce')
df['ADS_TPM'] = pd.to_numeric(df['ADS_TPM'], errors='coerce')

# ---------------- 2. 计算一阶差分与一阶导数 ----------------
# 按照生境和防御系统分组，并按时间先后排序
df = df.sort_values(by=['Environment', 'DS', 'Time_Parsed']).reset_index(drop=True)

# 计算时间差（天）
df['Delta_Days'] = df.groupby(['Environment', 'DS'])['Time_Parsed'].diff().dt.days

# 计算 TPM 差值（一阶差分）
df['Delta_DS_TPM'] = df.groupby(['Environment', 'DS'])['DS_TPM'].diff()
df['Delta_ADS_TPM'] = df.groupby(['Environment', 'DS'])['ADS_TPM'].diff()

# 计算一阶导数（变化率：TPM / 天）
# 考虑到可能有同一天取样导致除以 0，使用 np.where 保护
df['DS_Derivative'] = np.where(df['Delta_Days'] > 0, df['Delta_DS_TPM'] / df['Delta_Days'], np.nan)
df['ADS_Derivative'] = np.where(df['Delta_Days'] > 0, df['Delta_ADS_TPM'] / df['Delta_Days'], np.nan)

# 去除 NaN 值（第一次取样没有前置节点，无法计算差值）
df_diff = df.dropna(subset=['DS_Derivative']).copy()

print("--- 部分一阶导数计算结果 ---")
print(df_diff[['Environment', 'DS', 'Time', 'Delta_Days', 'DS_Derivative']].head())

# ---------------- 3. 计算不同生境间的一阶导相关性 ----------------
# 提取出所有的生境
environments = df_diff['Environment'].unique()

if len(environments) >= 2:
    # 策略：以防御系统 (Defense System) 为特征，计算每个生境中该系统的平均一阶导数
    # 如果你想分析的是 "不同环境防御系统的动态演化趋势是否一致"
    pivot_df = df_diff.groupby(['Environment', 'DS'])['DS_Derivative'].mean().unstack('Environment')

    # 丢弃在某些生境中缺失的防御系统
    pivot_df = pivot_df.dropna()

    print("\n--- 不同生境在相同防御系统上的平均一阶导数 (对齐后) ---")
    print(pivot_df)

    if len(pivot_df) >= 3: # 相关性计算至少需要3个数据点
        # 提取两个生境进行相关性计算 (以示例中的 spring 和 AS 为例)
        env1, env2 = environments[0], environments[1]

        # Pearson 检验线性相关性，Spearman 检验单调相关性（对极端异常值更稳健）
        pearson_corr, p_value_p = pearsonr(pivot_df[env1], pivot_df[env2])
        spearman_corr, p_value_s = spearmanr(pivot_df[env1], pivot_df[env2])

        print(f"\n--- {env1} vs {env2} 显著性检验结果 ---")
        print(f"Pearson相关系数: {pearson_corr:.4f}, P-value: {p_value_p:.4g}")
        print(f"Spearman相关系数: {spearman_corr:.4f}, P-value: {p_value_s:.4g}")

        if p_value_s < 0.05:
            print("结论: 两生境的防御系统动态演化趋势呈显著相关。")
        else:
            print("结论: 未发现显著相关性。")
    else:
        print("\n警告: 两个生境间共有的防御系统种类太少，无法进行有统计学意义的相关性计算。")
else:
    print("\n生境数量不足，无法进行跨生境相关性计算。")

0      spring
1      spring
2      spring
3      spring
4      spring
        ...  
484    sewage
485    sewage
486    sewage
487    sewage
488    sewage
Name: Environment, Length: 489, dtype: object
--- 部分一阶导数计算结果 ---
  Environment     DS    Time  Delta_Days  DS_Derivative
1          AS  CBASS  201302        31.0     222.548800
2          AS  CBASS  201303        28.0    -690.705514
3          AS  CBASS  201304        31.0      -6.257266
4          AS  CBASS  201305        30.0     -46.035632
5          AS  CBASS  201306        31.0     312.039238

--- 不同生境在相同防御系统上的平均一阶导数 (对齐后) ---
Environment         AS      marine  reservoir     sewage      spring
DS                                                                  
RM          -84.516811  234.135757  39.742977 -78.493985 -236.714746

警告: 两个生境间共有的防御系统种类太少，无法进行有统计学意义的相关性计算。


In [13]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, pearsonr
import matplotlib.pyplot as plt
import os

# ==========================================
# Parameters
# ==========================================

INPUT_FILE = r"D:\softerware_scientific_research\python_all_inone\py learning\pythonProject\prophage_project\huiyuan_ADS_DS\time_scale2.csv"

OUTPUT_DIR = r"D:\softerware_scientific_research\python_all_inone\py learning\pythonProject\prophage_project\huiyuan_ADS_DS\DS_ADS_analysis"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ==========================================
# 1. Load data
# ==========================================

df = pd.read_csv(INPUT_FILE)

# 删除完全为空的列
df = df.dropna(axis=1, how="all")

# 只保留前7列
df = df.iloc[:, :7]

df.columns = [
    "Environment",
    "Sample_ID",
    "Time",
    "DS",
    "DS_TPM",
    "ADS",
    "ADS_TPM"
]

print("Raw data:")
print(df.head())


# ==========================================
# 2. Convert TPM to numeric
# ==========================================

df["DS_TPM"] = pd.to_numeric(df["DS_TPM"], errors="coerce")
df["ADS_TPM"] = pd.to_numeric(df["ADS_TPM"], errors="coerce")


# ==========================================
# 3. Parse Time
# ==========================================

def parse_time(x):
    """
    自动解析不同时间格式，例如：

    2026/12/19
    2026/3/18
    201808
    201307
    2026-12-19
    """

    x = str(x).strip()

    # YYYYMM，例如 201808
    if len(x) == 6 and x.isdigit():

        return pd.to_datetime(
            x,
            format="%Y%m",
            errors="coerce"
        )

    # 常规格式
    return pd.to_datetime(
        x,
        errors="coerce"
    )


df["Time_parsed"] = df["Time"].apply(parse_time)


# 检查无法解析的时间
failed_time = df[df["Time_parsed"].isna()]

if len(failed_time) > 0:

    print("\nWarning: Some time values cannot be parsed:")

    print(
        failed_time["Time"]
        .drop_duplicates()
        .tolist()
    )


# ==========================================
# 4. Aggregate repeated observations
# ==========================================

# 同一个 Environment + Time + DS
# 如果存在多个样本，计算平均 TPM

ds_summary = (
    df.groupby(
        [
            "Environment",
            "Time_parsed",
            "DS"
        ],
        as_index=False
    )["DS_TPM"]
    .mean()
)


# 同一个 Environment + Time + ADS
# 如果存在多个样本，计算平均 TPM

ads_summary = (
    df.groupby(
        [
            "Environment",
            "Time_parsed",
            "ADS"
        ],
        as_index=False
    )["ADS_TPM"]
    .mean()
)


# ==========================================
# 5. Calculate first difference
# ==========================================

# DS排序
ds_summary = ds_summary.sort_values(
    [
        "Environment",
        "DS",
        "Time_parsed"
    ]
)

# ADS排序
ads_summary = ads_summary.sort_values(
    [
        "Environment",
        "ADS",
        "Time_parsed"
    ]
)


# 计算一阶差分
ds_summary["DS_delta"] = (
    ds_summary
    .groupby(
        [
            "Environment",
            "DS"
        ]
    )["DS_TPM"]
    .diff()
)


ads_summary["ADS_delta"] = (
    ads_summary
    .groupby(
        [
            "Environment",
            "ADS"
        ]
    )["ADS_TPM"]
    .diff()
)


# ==========================================
# 6. Optional:
# Calculate time-normalized first derivative
# ==========================================

# 计算时间间隔（天）

ds_summary["Time_interval_days"] = (
    ds_summary
    .groupby(
        [
            "Environment",
            "DS"
        ]
    )["Time_parsed"]
    .diff()
    .dt.days
)


ads_summary["Time_interval_days"] = (
    ads_summary
    .groupby(
        [
            "Environment",
            "ADS"
        ]
    )["Time_parsed"]
    .diff()
    .dt.days
)


# 防止除以0

ds_summary["DS_rate"] = (
    ds_summary["DS_delta"]
    /
    ds_summary["Time_interval_days"]
)


ads_summary["ADS_rate"] = (
    ads_summary["ADS_delta"]
    /
    ads_summary["Time_interval_days"]
)


# ==========================================
# 7. Save first difference results
# ==========================================

ds_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "DS_first_difference.csv"
    ),
    index=False
)


ads_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "ADS_first_difference.csv"
    ),
    index=False
)


# ==========================================
# 8. Match DS and ADS
# ==========================================

# DS -> ADS对应关系

ds_ads_mapping = {
    "Cas": "Anti_CRISPR",
    "RM": "Anti_RM",
    "CBASS": "Anti_CBASS",
    "Pycsar": "Anti_Pycsar",
    "Dnd": "Anti_Dnd",
    "Gabija": "Anti_Gabija",
    "Retron": "Anti_Retron",
    "Thoeris": "Anti_Thoeris"
}


# ==========================================
# 9. Correlation analysis
# ==========================================

results = []


for ds_name, ads_name in ds_ads_mapping.items():

    print(f"\nAnalyzing: {ds_name} vs {ads_name}")

    # 获取DS变化量

    ds_temp = ds_summary[
        ds_summary["DS"] == ds_name
    ][
        [
            "Environment",
            "Time_parsed",
            "DS_delta",
            "DS_rate"
        ]
    ].copy()


    # 获取ADS变化量

    ads_temp = ads_summary[
        ads_summary["ADS"] == ads_name
    ][
        [
            "Environment",
            "Time_parsed",
            "ADS_delta",
            "ADS_rate"
        ]
    ].copy()


    # 合并
    merged = pd.merge(
        ds_temp,
        ads_temp,
        on=[
            "Environment",
            "Time_parsed"
        ],
        how="inner"
    )


    # 删除第一个时间点产生的NaN

    merged_delta = merged.dropna(
        subset=[
            "DS_delta",
            "ADS_delta"
        ]
    )


    merged_rate = merged.dropna(
        subset=[
            "DS_rate",
            "ADS_rate"
        ]
    )


    # ======================================
    # Difference correlation
    # ======================================

    if len(merged_delta) >= 3:

        # Spearman
        spearman_r, spearman_p = spearmanr(
            merged_delta["DS_delta"],
            merged_delta["ADS_delta"]
        )

        # Pearson
        pearson_r, pearson_p = pearsonr(
            merged_delta["DS_delta"],
            merged_delta["ADS_delta"]
        )

    else:

        spearman_r = np.nan
        spearman_p = np.nan

        pearson_r = np.nan
        pearson_p = np.nan


    # ======================================
    # Time-normalized rate correlation
    # ======================================

    if len(merged_rate) >= 3:

        rate_spearman_r, rate_spearman_p = spearmanr(
            merged_rate["DS_rate"],
            merged_rate["ADS_rate"]
        )

        rate_pearson_r, rate_pearson_p = pearsonr(
            merged_rate["DS_rate"],
            merged_rate["ADS_rate"]
        )

    else:

        rate_spearman_r = np.nan
        rate_spearman_p = np.nan

        rate_pearson_r = np.nan
        rate_pearson_p = np.nan


    # ======================================
    # Save results
    # ======================================

    results.append({

        "DS": ds_name,

        "ADS": ads_name,

        "N_delta": len(merged_delta),

        "Spearman_r_delta": spearman_r,

        "Spearman_p_delta": spearman_p,

        "Pearson_r_delta": pearson_r,

        "Pearson_p_delta": pearson_p,

        "N_rate": len(merged_rate),

        "Spearman_r_rate": rate_spearman_r,

        "Spearman_p_rate": rate_spearman_p,

        "Pearson_r_rate": rate_pearson_r,

        "Pearson_p_rate": rate_pearson_p

    })


# ==========================================
# 10. Save correlation results
# ==========================================

results_df = pd.DataFrame(results)

results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "DS_ADS_correlation_results.csv"
    ),
    index=False
)


print("\n===================================")
print("Correlation Results")
print("===================================")

print(results_df)


# ==========================================
# 11. Plot scatterplots
# ==========================================

for ds_name, ads_name in ds_ads_mapping.items():

    ds_temp = ds_summary[
        ds_summary["DS"] == ds_name
    ][
        [
            "Environment",
            "Time_parsed",
            "DS_delta"
        ]
    ].copy()


    ads_temp = ads_summary[
        ads_summary["ADS"] == ads_name
    ][
        [
            "Environment",
            "Time_parsed",
            "ADS_delta"
        ]
    ].copy()


    merged = pd.merge(
        ds_temp,
        ads_temp,
        on=[
            "Environment",
            "Time_parsed"
        ],
        how="inner"
    )


    merged = merged.dropna(
        subset=[
            "DS_delta",
            "ADS_delta"
        ]
    )


    if len(merged) < 3:
        continue


    # Spearman correlation

    r, p = spearmanr(
        merged["DS_delta"],
        merged["ADS_delta"]
    )


    # 绘图

    plt.figure(
        figsize=(6, 5)
    )


    # 不同Environment使用不同颜色

    environments = merged["Environment"].unique()

    for env in environments:

        subset = merged[
            merged["Environment"] == env
        ]

        plt.scatter(
            subset["DS_delta"],
            subset["ADS_delta"],
            alpha=0.7,
            label=env
        )


    plt.axhline(
        0,
        linestyle="--",
        linewidth=1
    )

    plt.axvline(
        0,
        linestyle="--",
        linewidth=1
    )


    plt.xlabel(
        f"Δ {ds_name} TPM"
    )

    plt.ylabel(
        f"Δ {ads_name} TPM"
    )


    plt.title(
        f"{ds_name} vs {ads_name}\n"
        f"Spearman r = {r:.3f}, P = {p:.3e}"
    )


    plt.legend(
        title="Environment",
        bbox_to_anchor=(1.05, 1),
        loc="upper left"
    )


    plt.tight_layout()


    plt.savefig(
        os.path.join(
            OUTPUT_DIR,
            f"{ds_name}_vs_{ads_name}_delta.pdf"
        ),
        dpi=300
    )


    plt.close()


print("\nAnalysis finished!")

print(
    f"\nResults saved to:\n{OUTPUT_DIR}"
)

Raw data:
  Environment    Sample_ID        Time      DS     DS_TPM          ADS  \
0      spring  SRR21619491  2026/12/19     Cas  693303.00  Anti_CRISPR   
1      spring  SRR21619491  2026/12/19      RM  154392.00      Anti_RM   
2      spring  SRR21619491  2026/12/19   CBASS    1417.79   Anti_CBASS   
3      spring  SRR21619491  2026/12/19  Pycsar       0.00  Anti_Pycsar   
4      spring  SRR21619492   2026/3/18     Cas  629643.00  Anti_CRISPR   

      ADS_TPM  
0  26543.0000  
1   1020.6800  
2     84.9854  
3   1055.8800  
4  34967.5000  

Analyzing: Cas vs Anti_CRISPR

Analyzing: RM vs Anti_RM

Analyzing: CBASS vs Anti_CBASS

Analyzing: Pycsar vs Anti_Pycsar

Analyzing: Dnd vs Anti_Dnd

Analyzing: Gabija vs Anti_Gabija

Analyzing: Retron vs Anti_Retron

Analyzing: Thoeris vs Anti_Thoeris

Correlation Results
        DS           ADS  N_delta  Spearman_r_delta  Spearman_p_delta  \
0      Cas   Anti_CRISPR       60         -0.064466          0.624590   
1       RM       Anti_RM   

In [15]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, pearsonr
import matplotlib.pyplot as plt
import os

# ==========================================
# Parameters
# ==========================================

INPUT_FILE = r"D:\softerware_scientific_research\python_all_inone\py learning\pythonProject\prophage_project\huiyuan_ADS_DS\time_scale2.csv"
OUTPUT_DIR = r"D:\softerware_scientific_research\python_all_inone\py learning\pythonProject\prophage_project\huiyuan_ADS_DS\DS_ADS_analysis"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# 1. Load data
# ==========================================

df = pd.read_csv(INPUT_FILE)

# 删除完全为空的列
df = df.dropna(axis=1, how="all")

# 只保留前7列
df = df.iloc[:, :7]

df.columns = [
    "Environment",
    "Sample_ID",
    "Time",
    "DS",
    "DS_TPM",
    "ADS",
    "ADS_TPM"
]

print("Raw data:")
print(df.head())

# ==========================================
# 2. Convert TPM to numeric
# ==========================================

df["DS_TPM"] = pd.to_numeric(df["DS_TPM"], errors="coerce")
df["ADS_TPM"] = pd.to_numeric(df["ADS_TPM"], errors="coerce")

# ==========================================
# 3. Parse Time
# ==========================================

def parse_time(x):
    """自动解析不同时间格式"""
    x = str(x).strip()

    # YYYYMM，例如 201808
    if len(x) == 6 and x.isdigit():
        return pd.to_datetime(x, format="%Y%m", errors="coerce")

    # 常规格式
    return pd.to_datetime(x, errors="coerce")

df["Time_parsed"] = df["Time"].apply(parse_time)

# 检查无法解析的时间
failed_time = df[df["Time_parsed"].isna()]

if len(failed_time) > 0:
    print("\nWarning: Some time values cannot be parsed:")
    print(failed_time["Time"].drop_duplicates().tolist())

# ==========================================
# 4. Aggregate repeated observations
# ==========================================

# 同一个 Environment + Time + DS
ds_summary = (
    df.groupby(["Environment", "Time_parsed", "DS"], as_index=False)["DS_TPM"]
    .mean()
)

# 同一个 Environment + Time + ADS
ads_summary = (
    df.groupby(["Environment", "Time_parsed", "ADS"], as_index=False)["ADS_TPM"]
    .mean()
)

# ==========================================
# 5. Calculate first difference
# ==========================================

# DS排序
ds_summary = ds_summary.sort_values(["Environment", "DS", "Time_parsed"])

# ADS排序
ads_summary = ads_summary.sort_values(["Environment", "ADS", "Time_parsed"])

# 计算一阶差分
ds_summary["DS_delta"] = (
    ds_summary.groupby(["Environment", "DS"])["DS_TPM"].diff()
)

ads_summary["ADS_delta"] = (
    ads_summary.groupby(["Environment", "ADS"])["ADS_TPM"].diff()
)

# ==========================================
# 6. Optional: Calculate time-normalized first derivative
# ==========================================

# 计算时间间隔（天）
ds_summary["Time_interval_days"] = (
    ds_summary.groupby(["Environment", "DS"])["Time_parsed"].diff().dt.days
)

ads_summary["Time_interval_days"] = (
    ads_summary.groupby(["Environment", "ADS"])["Time_parsed"].diff().dt.days
)

# 防止除以0
ds_summary["DS_rate"] = (
    ds_summary["DS_delta"] / ds_summary["Time_interval_days"]
)

ads_summary["ADS_rate"] = (
    ads_summary["ADS_delta"] / ads_summary["Time_interval_days"]
)

# ==========================================
# 7. Save first difference results
# ==========================================

ds_summary.to_csv(
    os.path.join(OUTPUT_DIR, "DS_first_difference.csv"),
    index=False
)

ads_summary.to_csv(
    os.path.join(OUTPUT_DIR, "ADS_first_difference.csv"),
    index=False
)

# ==========================================
# 8. Match DS and ADS
# ==========================================

ds_ads_mapping = {
    "Cas": "Anti_CRISPR",
    "RM": "Anti_RM",
    "CBASS": "Anti_CBASS",
    "Pycsar": "Anti_Pycsar",
    "Dnd": "Anti_Dnd",
    "Gabija": "Anti_Gabija",
    "Retron": "Anti_Retron",
    "Thoeris": "Anti_Thoeris"
}

# ==========================================
# 9. Correlation analysis
# ==========================================

results = []

for ds_name, ads_name in ds_ads_mapping.items():
    print(f"\nAnalyzing: {ds_name} vs {ads_name}")

    ds_temp = ds_summary[ds_summary["DS"] == ds_name][
        ["Environment", "Time_parsed", "DS_delta", "DS_rate"]
    ].copy()

    ads_temp = ads_summary[ads_summary["ADS"] == ads_name][
        ["Environment", "Time_parsed", "ADS_delta", "ADS_rate"]
    ].copy()

    # 合并
    merged = pd.merge(
        ds_temp, ads_temp, on=["Environment", "Time_parsed"], how="inner"
    )

    # 删除NaN
    merged_delta = merged.dropna(subset=["DS_delta", "ADS_delta"])
    merged_rate = merged.dropna(subset=["DS_rate", "ADS_rate"])

    # ======================================
    # Difference correlation
    # ======================================
    if len(merged_delta) >= 3:
        spearman_r, spearman_p = spearmanr(merged_delta["DS_delta"], merged_delta["ADS_delta"])
        pearson_r, pearson_p = pearsonr(merged_delta["DS_delta"], merged_delta["ADS_delta"])
    else:
        spearman_r = spearman_p = np.nan
        pearson_r = pearson_p = np.nan

    # ======================================
    # Time-normalized rate correlation
    # ======================================
    if len(merged_rate) >= 3:
        rate_spearman_r, rate_spearman_p = spearmanr(merged_rate["DS_rate"], merged_rate["ADS_rate"])
        rate_pearson_r, rate_pearson_p = pearsonr(merged_rate["DS_rate"], merged_rate["ADS_rate"])
    else:
        rate_spearman_r = rate_spearman_p = np.nan
        rate_pearson_r = rate_pearson_p = np.nan

    # Save results
    results.append({
        "DS": ds_name,
        "ADS": ads_name,
        "N_delta": len(merged_delta),
        "Spearman_r_delta": spearman_r,
        "Spearman_p_delta": spearman_p,
        "Pearson_r_delta": pearson_r,
        "Pearson_p_delta": pearson_p,
        "N_rate": len(merged_rate),
        "Spearman_r_rate": rate_spearman_r,
        "Spearman_p_rate": rate_spearman_p,
        "Pearson_r_rate": rate_pearson_r,
        "Pearson_p_rate": rate_pearson_p
    })

# ==========================================
# 10. Save correlation results
# ==========================================

results_df = pd.DataFrame(results)
results_df.to_csv(
    os.path.join(OUTPUT_DIR, "DS_ADS_correlation_results.csv"),
    index=False
)

print("\n===================================")
print("Correlation Results")
print("===================================")
print(results_df)

# ==========================================
# 11. Plot scatterplots (分面图重构)
# ==========================================

# 确定网格尺寸，8 个关系对适合 2 行 4 列
n_pairs = len(ds_ads_mapping)
cols = 4
rows = (n_pairs + cols - 1) // cols

# 创建分面画布
fig, axes = plt.subplots(rows, cols, figsize=(18, 4 * rows))
axes = axes.flatten()

# 用于收集所有子图的图例对象，以便最后生成统一图例
legend_handles = {}

for idx, (ds_name, ads_name) in enumerate(ds_ads_mapping.items()):
    ax = axes[idx]

    ds_temp = ds_summary[ds_summary["DS"] == ds_name][
        ["Environment", "Time_parsed", "DS_delta"]
    ].copy()

    ads_temp = ads_summary[ads_summary["ADS"] == ads_name][
        ["Environment", "Time_parsed", "ADS_delta"]
    ].copy()

    merged = pd.merge(
        ds_temp, ads_temp, on=["Environment", "Time_parsed"], how="inner"
    )

    merged = merged.dropna(subset=["DS_delta", "ADS_delta"])

    if len(merged) < 3:
        ax.set_title(f"{ds_name} vs {ads_name}\nNot enough data")
        ax.axis("off")  # 数据不足时隐藏坐标轴
        continue

    # 计算 Spearman correlation
    r, p = spearmanr(merged["DS_delta"], merged["ADS_delta"])

    # 绘图
    environments = merged["Environment"].unique()

    for env in environments:
        subset = merged[merged["Environment"] == env]

        scatter = ax.scatter(
            subset["DS_delta"],
            subset["ADS_delta"],
            alpha=0.7,
            label=env
        )

        # 收集唯一环境的句柄和标签
        if env not in legend_handles:
            legend_handles[env] = scatter

    ax.axhline(0, linestyle="--", linewidth=1, color="gray")
    ax.axvline(0, linestyle="--", linewidth=1, color="gray")

    ax.set_xlabel(f"Δ {ds_name} TPM")
    ax.set_ylabel(f"Δ {ads_name} TPM")
    ax.set_title(f"{ds_name} vs {ads_name}\nSpearman r = {r:.3f}, P = {p:.3e}")

# 隐藏多余的子图（如果有）
for i in range(idx + 1, len(axes)):
    axes[i].axis("off")

# 创建统一图例并放置在画布右侧外围
if legend_handles:
    fig.legend(
        legend_handles.values(),
        legend_handles.keys(),
        title="Environment",
        bbox_to_anchor=(1.01, 0.5),
        loc="center left"
    )

plt.tight_layout()

# 保存为单张 PDF
plt.savefig(
    os.path.join(OUTPUT_DIR, "DS_vs_ADS_delta_faceted.pdf"),
    dpi=300,
    bbox_inches="tight" # 确保外部图例不会被裁剪
)

plt.close()

print("\nAnalysis finished!")
print(f"\nResults saved to:\n{OUTPUT_DIR}")

Raw data:
  Environment    Sample_ID       Time      DS     DS_TPM          ADS  \
0      spring  SRR21619491  2019/12/1     Cas  693303.00  Anti_CRISPR   
1      spring  SRR21619491  2019/12/1      RM  154392.00      Anti_RM   
2      spring  SRR21619491  2019/12/1   CBASS    1417.79   Anti_CBASS   
3      spring  SRR21619491  2019/12/1  Pycsar       0.00  Anti_Pycsar   
4      spring  SRR21619492   2018/3/1     Cas  629643.00  Anti_CRISPR   

      ADS_TPM  
0  26543.0000  
1   1020.6800  
2     84.9854  
3   1055.8800  
4  34967.5000  

Analyzing: Cas vs Anti_CRISPR

Analyzing: RM vs Anti_RM

Analyzing: CBASS vs Anti_CBASS

Analyzing: Pycsar vs Anti_Pycsar

Analyzing: Dnd vs Anti_Dnd

Analyzing: Gabija vs Anti_Gabija

Analyzing: Retron vs Anti_Retron

Analyzing: Thoeris vs Anti_Thoeris

Correlation Results
        DS           ADS  N_delta  Spearman_r_delta  Spearman_p_delta  \
0      Cas   Anti_CRISPR       60         -0.023008          0.861480   
1       RM       Anti_RM       80

In [3]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, pearsonr
import matplotlib.pyplot as plt
import os
import matplotlib.colors as mcolors
import seaborn as sns

# ==========================================
# Parameters
# ==========================================

INPUT_FILE = r"D:\softerware_scientific_research\python_all_inone\py learning\pythonProject\prophage_project\huiyuan_ADS_DS\time_scale2.csv"

# 建议修改输出目录名称以区分旧结果
OUTPUT_DIR = r"D:\softerware_scientific_research\python_all_inone\py learning\pythonProject\prophage_project\huiyuan_ADS_DS\DS_ADS_analysis_CLR_FreshColor_environment"

# CLR transformation pseudocount
PSEUDOCOUNT = 1

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ==========================================
# 1. Load data
# ==========================================
print("Loading data...")
df = pd.read_csv(INPUT_FILE)

# 删除完全为空的列
df = df.dropna(axis=1, how="all")

# 只保留前7列
df = df.iloc[:, :7]

df.columns = [
    "Environment",
    "Sample_ID",
    "Time",
    "DS",
    "DS_TPM",
    "ADS",
    "ADS_TPM"
]

# ==========================================
# 2. Convert TPM to numeric
# ==========================================

df["DS_TPM"] = pd.to_numeric(
    df["DS_TPM"],
    errors="coerce"
)

df["ADS_TPM"] = pd.to_numeric(
    df["ADS_TPM"],
    errors="coerce"
)


# ==========================================
# 3. Parse Time
# ==========================================

def parse_time(x):
    """自动解析不同时间格式"""

    x = str(x).strip()

    # YYYYMM，例如 201808
    if len(x) == 6 and x.isdigit():

        return pd.to_datetime(
            x,
            format="%Y%m",
            errors="coerce"
        )

    # 常规格式
    return pd.to_datetime(
        x,
        errors="coerce"
    )


df["Time_parsed"] = df["Time"].apply(
    parse_time
)


# 检查无法解析的时间

failed_time = df[
    df["Time_parsed"].isna()
]

if len(failed_time) > 0:

    print(
        "\nWarning: Some time values cannot be parsed:"
    )

    print(
        failed_time["Time"]
        .drop_duplicates()
        .tolist()
    )


# ==========================================
# 4. Aggregate repeated observations
# ==========================================

# 同一个 Environment + Time + DS
# 计算平均 TPM

ds_summary = (
    df.groupby(
        [
            "Environment",
            "Time_parsed",
            "DS"
        ],
        as_index=False
    )["DS_TPM"]
    .mean()
)


# 同一个 Environment + Time + ADS
# 计算平均 TPM

ads_summary = (
    df.groupby(
        [
            "Environment",
            "Time_parsed",
            "ADS"
        ],
        as_index=False
    )["ADS_TPM"]
    .mean()
)


# ==========================================
# 5. CLR transformation
# ==========================================

def clr_transform(series, pseudocount=1):
    """
    CLR transformation
    """

    x = series + pseudocount

    log_x = np.log(x)

    clr_x = log_x - log_x.mean()

    return clr_x


# ------------------------------------------
# DS CLR transformation
# 每个 Environment + Time 内部进行 CLR
# ------------------------------------------

ds_summary["DS_CLR"] = (
    ds_summary
    .groupby(
        [
            "Environment",
            "Time_parsed"
        ]
    )["DS_TPM"]
    .transform(
        lambda x: clr_transform(
            x,
            pseudocount=PSEUDOCOUNT
        )
    )
)


# ------------------------------------------
# ADS CLR transformation
# 每个 Environment + Time 内部进行 CLR
# ------------------------------------------

ads_summary["ADS_CLR"] = (
    ads_summary
    .groupby(
        [
            "Environment",
            "Time_parsed"
        ]
    )["ADS_TPM"]
    .transform(
        lambda x: clr_transform(
            x,
            pseudocount=PSEUDOCOUNT
        )
    )
)


# ==========================================
# 6. Sort data
# ==========================================

ds_summary = ds_summary.sort_values(
    [
        "Environment",
        "DS",
        "Time_parsed"
    ]
)

ads_summary = ads_summary.sort_values(
    [
        "Environment",
        "ADS",
        "Time_parsed"
    ]
)


# ==========================================
# 7. Calculate first difference
# ==========================================

# ------------------------------------------
# 原始 TPM 一阶差分
# ------------------------------------------

ds_summary["DS_delta"] = (
    ds_summary
    .groupby(
        [
            "Environment",
            "DS"
        ]
    )["DS_TPM"]
    .diff()
)


ads_summary["ADS_delta"] = (
    ads_summary
    .groupby(
        [
            "Environment",
            "ADS"
        ]
    )["ADS_TPM"]
    .diff()
)


# ------------------------------------------
# CLR 一阶差分
# ------------------------------------------

ds_summary["DS_CLR_delta"] = (
    ds_summary
    .groupby(
        [
            "Environment",
            "DS"
        ]
    )["DS_CLR"]
    .diff()
)


ads_summary["ADS_CLR_delta"] = (
    ads_summary
    .groupby(
        [
            "Environment",
            "ADS"
        ]
    )["ADS_CLR"]
    .diff()
)


# ==========================================
# 8. Calculate time-normalized rate
# ==========================================

# ------------------------------------------
# DS 时间间隔
# ------------------------------------------

ds_summary["Time_interval_days"] = (
    ds_summary
    .groupby(
        [
            "Environment",
            "DS"
        ]
    )["Time_parsed"]
    .diff()
    .dt.days
)


# ------------------------------------------
# ADS 时间间隔
# ------------------------------------------

ads_summary["Time_interval_days"] = (
    ads_summary
    .groupby(
        [
            "Environment",
            "ADS"
        ]
    )["Time_parsed"]
    .diff()
    .dt.days
)


# ------------------------------------------
# 原始 TPM rate
# ------------------------------------------

ds_summary["DS_rate"] = (
    ds_summary["DS_delta"]
    /
    ds_summary["Time_interval_days"]
)


ads_summary["ADS_rate"] = (
    ads_summary["ADS_delta"]
    /
    ads_summary["Time_interval_days"]
)


# ------------------------------------------
# CLR rate
# ------------------------------------------

ds_summary["DS_CLR_rate"] = (
    ds_summary["DS_CLR_delta"]
    /
    ds_summary["Time_interval_days"]
)


ads_summary["ADS_CLR_rate"] = (
    ads_summary["ADS_CLR_delta"]
    /
    ads_summary["Time_interval_days"]
)


# ==========================================
# 9. Save processed data
# ==========================================
print("Saving processed data...")
ds_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "DS_first_difference_with_CLR.csv"
    ),
    index=False
)


ads_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "ADS_first_difference_with_CLR.csv"
    ),
    index=False
)


# ==========================================
# 10. Match DS and ADS
# ==========================================

ds_ads_mapping = {
    "Cas": "Anti_CRISPR",
    "RM": "Anti_RM",
    "CBASS": "Anti_CBASS",
    "Pycsar": "Anti_Pycsar",
    "Dnd": "Anti_Dnd",
    "Gabija": "Anti_Gabija",
    "Retron": "Anti_Retron",
    "Thoeris": "Anti_Thoeris"
}

# ==========================================
# 11. Calculate Correlations (Overall & Per-Environment)
# ==========================================
print("Calculating correlations...")

results = []

# [新增功能开关]：是否分 Environment 分别进行 Spearman 检验
CALCULATE_PER_ENV = True

for ds_name, ads_name in ds_ads_mapping.items():

    # 提取DS数据
    ds_temp = ds_summary[
        ds_summary["DS"] == ds_name
    ][["Environment", "Time_parsed", "DS_CLR_delta"]].copy()

    # 提取ADS数据
    ads_temp = ads_summary[
        ads_summary["ADS"] == ads_name
    ][["Environment", "Time_parsed", "ADS_CLR_delta"]].copy()

    # 根据 Environment 和 Time_parsed 进行合并
    merged = pd.merge(
        ds_temp, ads_temp,
        on=["Environment", "Time_parsed"],
        how="inner"
    )

    # 去除缺失值
    merged = merged.dropna(subset=["DS_CLR_delta", "ADS_CLR_delta"])

    # --- 1. 计算全局 (Overall) 相关性 ---
    if len(merged) >= 3:
        r_all, p_all = spearmanr(merged["DS_CLR_delta"], merged["ADS_CLR_delta"])
        results.append({
            "DS": ds_name,
            "ADS": ads_name,
            "Environment": "Overall", # 标记为所有环境合并
            "Spearman_r": r_all,
            "Spearman_P": p_all,
            "Sample_Size": len(merged)
        })
    else:
        results.append({
            "DS": ds_name, "ADS": ads_name, "Environment": "Overall",
            "Spearman_r": np.nan, "Spearman_P": np.nan, "Sample_Size": len(merged)
        })

    # --- 2. [新增] 计算各环境 (Per-Environment) 相关性 ---
    if CALCULATE_PER_ENV:
        environments_present = merged["Environment"].unique()

        for env in environments_present:
            subset = merged[merged["Environment"] == env]

            # Spearman 检验通常要求样本量 >= 3
            if len(subset) >= 3:
                r_env, p_env = spearmanr(subset["DS_CLR_delta"], subset["ADS_CLR_delta"])
            else:
                # 样本量不足时返回 NaN
                r_env, p_env = np.nan, np.nan

            results.append({
                "DS": ds_name,
                "ADS": ads_name,
                "Environment": env, # 具体的环境名称
                "Spearman_r": r_env,
                "Spearman_P": p_env,
                "Sample_Size": len(subset)
            })

# ==========================================
# 12. Save correlation results
# ==========================================
print("Saving correlation results...")
results_df = pd.DataFrame(results)

# 将全局结果和分环境结果保存到同一个 CSV 文件中
results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "DS_ADS_correlation_results_with_CLR.csv"
    ),
    index=False
)

print("Correlation results saved successfully.")

# ==========================================
# 13. Plot CLR difference scatterplots (保持你原来的代码不变)
# ==========================================




# ==========================================
# 12. Save correlation results
# ==========================================
print("Saving correlation results...")
results_df = pd.DataFrame(results)

results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "DS_ADS_correlation_results_with_CLR.csv"
    ),
    index=False
)

# 请确保在文件最上方补充此导入
# import seaborn as sns

# ==========================================
# 12.5 Plot Correlation Heatmap
# ==========================================
print("Generating correlation heatmap...")

# 将长表格转换为适合画热图的宽表格 (Pivot)
# 1. 提取 Spearman r 矩阵
pivot_r = results_df.pivot(
    index="DS",
    columns="Environment",
    values="Spearman_r"
)

# 2. 提取 Spearman P 矩阵
pivot_p = results_df.pivot(
    index="DS",
    columns="Environment",
    values="Spearman_P"
)

# 整理列的顺序：让 'Overall' 排在最前面，其他环境按字母排序
if "Overall" in pivot_r.columns:
    envs = [col for col in pivot_r.columns if col != "Overall"]
    sorted_cols = ["Overall"] + sorted(envs)
    pivot_r = pivot_r[sorted_cols]
    pivot_p = pivot_p[sorted_cols]

# 构建注释矩阵（在热图格子里显示 r 值及显著性星号）
annot_matrix = np.empty_like(pivot_r, dtype=object)

for i in range(pivot_r.shape[0]):
    for j in range(pivot_r.shape[1]):
        r_val = pivot_r.iloc[i, j]
        p_val = pivot_p.iloc[i, j]

        # 针对由于样本量不足(<3)导致的 NaN 进行处理
        if pd.isna(r_val) or pd.isna(p_val):
            annot_matrix[i, j] = "NaN"
        else:
            # 严谨的科学标注：根据 P 值添加显著性星号
            stars = ""
            if p_val < 0.001:
                stars = "***"
            elif p_val < 0.01:
                stars = "**"
            elif p_val < 0.05:
                stars = "*"

            annot_matrix[i, j] = f"{r_val:.2f}{stars}"

# 设置画布大小（可根据环境种类的多少自动调整宽度）
fig_width = max(8, len(pivot_r.columns) * 1.2)
fig, ax = plt.subplots(figsize=(fig_width, 6))

# 使用 seaborn 绘制热图
# cmap="RdBu_r" 是一种科学上常用的发散型色带，红色代表正相关，蓝色代表负相关，白色为0
sns.heatmap(
    pivot_r,
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    annot=annot_matrix,
    fmt="", # fmt="" 是必需的，因为 annot 包含字符串(星号)
    cbar_kws={'label': 'Spearman $r$'},
    ax=ax,
    linewidths=0.5,
    linecolor='white',
    annot_kws={"size": 10} # 调整格子内文字大小
)

ax.set_title(
    "Spearman Correlation between DS and ADS ($\Delta$ CLR)\n"
    "* $P < 0.05$, ** $P < 0.01$, *** $P < 0.001$",
    fontsize=12,
    pad=15
)
ax.set_xlabel("Environment", fontsize=11)
ax.set_ylabel("Defense System (DS)", fontsize=11)

# 调整 y 轴标签方向以便阅读
plt.yticks(rotation=0)

# 保存热图
heatmap_path = os.path.join(
    OUTPUT_DIR,
    "DS_ADS_correlation_heatmap.pdf"
)
plt.savefig(
    heatmap_path,
    dpi=300,
    bbox_inches="tight"
)
plt.close()

print(f"Heatmap saved to:\n{heatmap_path}")

# ==========================================
# 13. Plot CLR difference scatterplots (重点修改部分)
# ==========================================
print("Generating faceted scatterplot with fresh colors...")

# --- 核心修改：定义清新的、高区分度的颜色映射 ---

# 获取所有独特的 Environment 名称，确保映射覆盖所有数据
all_environments = sorted(df["Environment"].dropna().unique())

# 定义一个清新的定性调色板 (来源于 Colorbrewer Set2)
# Set2 非常适合 3-8 种类型的区分，颜色清新柔和
fresh_palette = [
    "#66c2a5", # 碧绿色 (Teal/Green) -> 建议给 Marine
    "#fc8d62", # 橙红色 (Orange) -> 建议给 Reservoir
    "#8da0cb", # 浅蓝色 (Blue)
    "#e78ac3", # 粉红色 (Pink)
    "#a6d854", # 黄绿色 (Lime Green)
    "#ffd92f", # 黄色 (Yellow)
    "#e5c494", # 浅褐色 (Tan)
    "#b3b3b3"  # 灰色 (Gray)
]

# 创建 Environment -> Color 的映射字典
env_color_map = {}

# 1. 显式指定 Reservoir 和 Marine 使用反差大的清新颜色
# 假设你的数据中 Reservoir 和 Marine 的准确拼写如下，请根据实际情况调整
target_reservoir = "reservoir"
target_marine = "marine"

# 安全地分配颜色，防止数据中不存在这两个 Environment
assigned_colors_idx = []

if target_marine in all_environments:
    env_color_map[target_marine] = fresh_palette[0] # Marine 用碧绿色
    assigned_colors_idx.append(0)

if target_reservoir in all_environments:
    env_color_map[target_reservoir] = fresh_palette[1] # Reservoir 用橙红色，反差大
    assigned_colors_idx.append(1)

# 2. 为剩余的 Environment 自动分配调色板中的其他颜色
current_palette_idx = 0
for env in all_environments:
    if env not in env_color_map:
        # 跳过已被占用的颜色索引
        while current_palette_idx in assigned_colors_idx and current_palette_idx < len(fresh_palette):
            current_palette_idx += 1

        # 如果还有颜色可用，则分配
        if current_palette_idx < len(fresh_palette):
            env_color_map[env] = fresh_palette[current_palette_idx]
            assigned_colors_idx.append(current_palette_idx)
        else:
            # 如果 Environment 数量超过调色板数量，回退到默认颜色（虽然很少见）
            env_color_map[env] = "black"

print(f"Using Color Mapping: {env_color_map}")

# --- 绘图循环部分 ---

n_pairs = len(ds_ads_mapping)
cols = 4
rows = (n_pairs + cols - 1) // cols

fig, axes = plt.subplots(
    rows,
    cols,
    figsize=(18, 4 * rows)
)

axes = axes.flatten()

# 用于收集全局图例的句柄
global_handles = []
global_labels = []

for idx, (ds_name, ads_name) in enumerate(ds_ads_mapping.items()):
    ax = axes[idx]

    # ... (数据提取和合并部分未变动) ...
    ds_temp = ds_summary[
        ds_summary["DS"] == ds_name
    ][["Environment", "Time_parsed", "DS_CLR_delta"]].copy()

    ads_temp = ads_summary[
        ads_summary["ADS"] == ads_name
    ][["Environment", "Time_parsed", "ADS_CLR_delta"]].copy()

    merged = pd.merge(
        ds_temp, ads_temp,
        on=["Environment", "Time_parsed"],
        how="inner"
    )

    merged = merged.dropna(
        subset=["DS_CLR_delta", "ADS_CLR_delta"]
    )

    if len(merged) < 3:
        ax.set_title(f"{ds_name} vs {ads_name}\nNot enough data", fontsize=10)
        ax.axis("off")
        continue

    # Spearman
    r, p = spearmanr(
        merged["DS_CLR_delta"],
        merged["ADS_CLR_delta"]
    )

    # --------------------------------------
    # Scatter (修改绘图颜色调用)
    # --------------------------------------
    environments_in_subplot = merged["Environment"].unique()

    for env in environments_in_subplot:
        subset = merged[merged["Environment"] == env]

        # --- 修改：显式调用映射的颜色 ---
        # 如果 env 不在 map 中（理论上不会发生），回退到灰色
        plot_color = env_color_map.get(env, "#b3b3b3")

        scatter = ax.scatter(
            subset["DS_CLR_delta"],
            subset["ADS_CLR_delta"],
            alpha=0.8, # 稍微调高透明度，颜色更饱满一点
            color=plot_color, # 显式指定颜色
            label=env,
            edgecolors='none', # 去掉散点边框，风格更清新现代
            s=40 # 稍微调整散点大小
        )

        # 收集用于全局图例的句柄（只收集独特的标签）
        if env not in global_labels:
            global_handles.append(scatter)
            global_labels.append(env)

    # --------------------------------------
    # 装饰 (略微调整样式以配合清新风格)
    # --------------------------------------
    ax.axhline(0, linestyle="-", linewidth=0.5, color="#d9d9d9") # 使用更浅的灰色实线作为参考线
    ax.axvline(0, linestyle="-", linewidth=0.5, color="#d9d9d9")

    ax.set_xlabel(f"$\Delta$ CLR({ds_name})", fontsize=9)
    ax.set_ylabel(f"$\Delta$ CLR({ads_name})", fontsize=9)

    # 处理 P 值的科学计数法显示
    p_str = f"{p:.3e}"
    if 'e' in p_str:
        base, exponent = p_str.split('e')
        p_label = f"{base} \\times 10^{{{int(exponent)}}}"
    else:
        p_label = p_str

    ax.set_title(
        f"{ds_name} vs {ads_name}\n"
        f"Spearman $r$ = {r:.3f}, $P$ = ${p_label}$",
        fontsize=10
    )

    # 调整坐标轴刻度标签大小
    ax.tick_params(axis='both', which='major', labelsize=8)


# ==========================================
# 隐藏多余的子图
# ==========================================
for i in range(idx + 1, len(axes)):
    axes[i].axis("off")


# ==========================================
# Unified legend (修改为紧凑布局)
# ==========================================
if global_handles:
    # 对图例进行排序，保证 reservoir 和 marine 在前面，或者按字母排序
    sorted_indices = np.argsort(global_labels)
    sorted_handles = [global_handles[i] for i in sorted_indices]
    sorted_labels = [global_labels[i] for i in sorted_indices]

    fig.legend(
        sorted_handles,
        sorted_labels,
        title="Environment",
        title_fontsize=11,
        fontsize=10,
        # 【核心修改 1】将 1.01 改为 0.92，让图例水平起始位置靠近子图的右边界(0.90)
        bbox_to_anchor=(0.92, 0.5),
        loc="center left",
        frameon=True,
        facecolor='white',
        edgecolor='#d9d9d9'
    )

# 建议使用 rect 参数直接限制 tight_layout 的作用范围，代替 subplots_adjust
# 这样不仅能精确控制右侧留白，还能防止上下标签被遮挡
# rect=[left, bottom, right, top]
# 【核心修改 2】让 tight_layout 只处理 0 到 0.90 的区域，将 0.90 到 1.0 的区域完美留给图例
plt.tight_layout(rect=[0, 0, 0.90, 1])

# 删除了原来的 plt.subplots_adjust(right=0.90)，因为 tight_layout(rect=...) 已经包含了这个功能且更稳健

# ==========================================
# Save figure
# ==========================================
plot_path = os.path.join(
    OUTPUT_DIR,
    "DS_vs_ADS_CLR_delta_faceted_fresh_colors.pdf"
)
plt.savefig(
    plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("\nAnalysis finished!")
print(f"\nResults saved to:\n{OUTPUT_DIR}")
print(f"Plot saved to:\n{plot_path}")

<>:584: SyntaxWarning: invalid escape sequence '\D'
<>:753: SyntaxWarning: invalid escape sequence '\D'
<>:754: SyntaxWarning: invalid escape sequence '\D'
<>:584: SyntaxWarning: invalid escape sequence '\D'
<>:753: SyntaxWarning: invalid escape sequence '\D'
<>:754: SyntaxWarning: invalid escape sequence '\D'
C:\Users\jingqing\AppData\Local\Temp\ipykernel_35808\4036801009.py:584: SyntaxWarning: invalid escape sequence '\D'
  "Spearman Correlation between DS and ADS ($\Delta$ CLR)\n"
C:\Users\jingqing\AppData\Local\Temp\ipykernel_35808\4036801009.py:753: SyntaxWarning: invalid escape sequence '\D'
  ax.set_xlabel(f"$\Delta$ CLR({ds_name})", fontsize=9)
C:\Users\jingqing\AppData\Local\Temp\ipykernel_35808\4036801009.py:754: SyntaxWarning: invalid escape sequence '\D'
  ax.set_ylabel(f"$\Delta$ CLR({ads_name})", fontsize=9)
C:\Users\jingqing\AppData\Local\Temp\ipykernel_35808\4036801009.py:458: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.


Loading data...
Saving processed data...
Calculating correlations...
Saving correlation results...
Correlation results saved successfully.
Saving correlation results...
Generating correlation heatmap...
Heatmap saved to:
D:\softerware_scientific_research\python_all_inone\py learning\pythonProject\prophage_project\huiyuan_ADS_DS\DS_ADS_analysis_CLR_FreshColor_environment\DS_ADS_correlation_heatmap.pdf
Generating faceted scatterplot with fresh colors...
Using Color Mapping: {'marine': '#66c2a5', 'reservoir': '#fc8d62', 'AS': '#8da0cb', 'sewage': '#e78ac3', 'spring': '#a6d854'}

Analysis finished!

Results saved to:
D:\softerware_scientific_research\python_all_inone\py learning\pythonProject\prophage_project\huiyuan_ADS_DS\DS_ADS_analysis_CLR_FreshColor_environment
Plot saved to:
D:\softerware_scientific_research\python_all_inone\py learning\pythonProject\prophage_project\huiyuan_ADS_DS\DS_ADS_analysis_CLR_FreshColor_environment\DS_vs_ADS_CLR_delta_faceted_fresh_colors.pdf


In [2]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, pearsonr
import matplotlib.pyplot as plt
import os

# ==========================================
# Parameters
# ==========================================

INPUT_FILE = r" time_scale2.csv"

OUTPUT_DIR = r"D:\softerware_scientific_research\python_all_inone\py learning\pythonProject\prophage_project\huiyuan_ADS_DS\DS_ADS_analysis_Lag_ADS"

# CLR transformation pseudocount
PSEUDOCOUNT = 1

# ==========================================
# Lag analysis parameters
# ==========================================

# 可选：
# "DS"  -> DS 作为领先变量
# "ADS" -> ADS 作为领先变量
INDEPENDENT_VARIABLE = "ADS"

# 最大测试 lag
# lag = 0：同步变化
# lag = 1：一个采样周期之后
# lag = 2：两个采样周期之后
MAX_LAG = 3


os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ==========================================
# 1. Load data
# ==========================================

print("Loading data...")

df = pd.read_csv(INPUT_FILE)

# 删除完全为空的列
df = df.dropna(
    axis=1,
    how="all"
)

# 只保留前7列
df = df.iloc[:, :7]

df.columns = [
    "Environment",
    "Sample_ID",
    "Time",
    "DS",
    "DS_TPM",
    "ADS",
    "ADS_TPM"
]


# ==========================================
# 2. Convert TPM to numeric
# ==========================================

df["DS_TPM"] = pd.to_numeric(
    df["DS_TPM"],
    errors="coerce"
)

df["ADS_TPM"] = pd.to_numeric(
    df["ADS_TPM"],
    errors="coerce"
)


# ==========================================
# 3. Parse Time
# ==========================================

def parse_time(x):

    """
    自动解析不同时间格式

    支持：
    2026/12/19
    2026-12-19
    201808
    """

    x = str(x).strip()

    # YYYYMM
    if len(x) == 6 and x.isdigit():

        return pd.to_datetime(
            x,
            format="%Y%m",
            errors="coerce"
        )

    # 常规日期格式
    return pd.to_datetime(
        x,
        errors="coerce"
    )


df["Time_parsed"] = df["Time"].apply(
    parse_time
)


# 检查无法解析的时间

failed_time = df[
    df["Time_parsed"].isna()
]

if len(failed_time) > 0:

    print(
        "\nWarning: Some time values cannot be parsed:"
    )

    print(
        failed_time["Time"]
        .drop_duplicates()
        .tolist()
    )


# ==========================================
# 删除无法解析时间的数据
# ==========================================

df = df.dropna(
    subset=["Time_parsed"]
)


# ==========================================
# 4. Aggregate repeated observations
# ==========================================

# ------------------------------------------
# DS
# ------------------------------------------

ds_summary = (
    df.groupby(
        [
            "Environment",
            "Time_parsed",
            "DS"
        ],
        as_index=False
    )["DS_TPM"]
    .mean()
)


# ------------------------------------------
# ADS
# ------------------------------------------

ads_summary = (
    df.groupby(
        [
            "Environment",
            "Time_parsed",
            "ADS"
        ],
        as_index=False
    )["ADS_TPM"]
    .mean()
)


# ==========================================
# 5. CLR transformation
# ==========================================

def clr_transform(
    series,
    pseudocount=1
):

    """
    CLR transformation

    CLR(x_i) =
    log(x_i + pseudocount)
    -
    mean(log(x + pseudocount))
    """

    x = series + pseudocount

    log_x = np.log(x)

    clr_x = log_x - log_x.mean()

    return clr_x


# ------------------------------------------
# DS CLR
# 每个 Environment + Time 内部
# ------------------------------------------

ds_summary["DS_CLR"] = (
    ds_summary
    .groupby(
        [
            "Environment",
            "Time_parsed"
        ]
    )["DS_TPM"]
    .transform(
        lambda x: clr_transform(
            x,
            pseudocount=PSEUDOCOUNT
        )
    )
)


# ------------------------------------------
# ADS CLR
# 每个 Environment + Time 内部
# ------------------------------------------

ads_summary["ADS_CLR"] = (
    ads_summary
    .groupby(
        [
            "Environment",
            "Time_parsed"
        ]
    )["ADS_TPM"]
    .transform(
        lambda x: clr_transform(
            x,
            pseudocount=PSEUDOCOUNT
        )
    )
)


# ==========================================
# 6. Sort data
# ==========================================

ds_summary = ds_summary.sort_values(
    [
        "Environment",
        "DS",
        "Time_parsed"
    ]
).reset_index(
    drop=True
)


ads_summary = ads_summary.sort_values(
    [
        "Environment",
        "ADS",
        "Time_parsed"
    ]
).reset_index(
    drop=True
)


# ==========================================
# 7. Calculate first difference
# ==========================================

# ------------------------------------------
# Raw TPM difference
# ------------------------------------------

ds_summary["DS_delta"] = (
    ds_summary
    .groupby(
        [
            "Environment",
            "DS"
        ]
    )["DS_TPM"]
    .diff()
)


ads_summary["ADS_delta"] = (
    ads_summary
    .groupby(
        [
            "Environment",
            "ADS"
        ]
    )["ADS_TPM"]
    .diff()
)


# ------------------------------------------
# CLR difference
# ------------------------------------------

ds_summary["DS_CLR_delta"] = (
    ds_summary
    .groupby(
        [
            "Environment",
            "DS"
        ]
    )["DS_CLR"]
    .diff()
)


ads_summary["ADS_CLR_delta"] = (
    ads_summary
    .groupby(
        [
            "Environment",
            "ADS"
        ]
    )["ADS_CLR"]
    .diff()
)


# ==========================================
# 8. Calculate time-normalized rate
# ==========================================

# ------------------------------------------
# DS time interval
# ------------------------------------------

ds_summary["Time_interval_days"] = (
    ds_summary
    .groupby(
        [
            "Environment",
            "DS"
        ]
    )["Time_parsed"]
    .diff()
    .dt.days
)


# ------------------------------------------
# ADS time interval
# ------------------------------------------

ads_summary["Time_interval_days"] = (
    ads_summary
    .groupby(
        [
            "Environment",
            "ADS"
        ]
    )["Time_parsed"]
    .diff()
    .dt.days
)


# 防止除以0

ds_summary.loc[
    ds_summary["Time_interval_days"] == 0,
    "Time_interval_days"
] = np.nan


ads_summary.loc[
    ads_summary["Time_interval_days"] == 0,
    "Time_interval_days"
] = np.nan


# ------------------------------------------
# Raw TPM rate
# ------------------------------------------

ds_summary["DS_rate"] = (
    ds_summary["DS_delta"]
    /
    ds_summary["Time_interval_days"]
)


ads_summary["ADS_rate"] = (
    ads_summary["ADS_delta"]
    /
    ads_summary["Time_interval_days"]
)


# ------------------------------------------
# CLR rate
# ------------------------------------------

ds_summary["DS_CLR_rate"] = (
    ds_summary["DS_CLR_delta"]
    /
    ds_summary["Time_interval_days"]
)


ads_summary["ADS_CLR_rate"] = (
    ads_summary["ADS_CLR_delta"]
    /
    ads_summary["Time_interval_days"]
)


# ==========================================
# 9. Save processed data
# ==========================================

print("Saving processed data...")


ds_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "DS_processed.csv"
    ),
    index=False
)


ads_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "ADS_processed.csv"
    ),
    index=False
)


# ==========================================
# 10. Match DS and ADS
# ==========================================

ds_ads_mapping = {
    "Cas": "Anti_CRISPR",
    "RM": "Anti_RM",
    "CBASS": "Anti_CBASS",
    "Pycsar": "Anti_Pycsar",
    "Dnd": "Anti_Dnd",
    "Gabija": "Anti_Gabija",
    "Retron": "Anti_Retron",
    "Thoeris": "Anti_Thoeris"
}


# ==========================================
# 11. Function: correlation calculation
# ==========================================

def calculate_correlation(
    data,
    x_col,
    y_col
):

    """
    计算 Spearman 和 Pearson correlation
    """

    temp = data.dropna(
        subset=[
            x_col,
            y_col
        ]
    )

    n = len(temp)

    if n >= 3:

        # 如果其中一个变量没有变化
        if (
            temp[x_col].nunique() < 2
            or
            temp[y_col].nunique() < 2
        ):

            return (
                n,
                np.nan,
                np.nan,
                np.nan,
                np.nan
            )

        spearman_r, spearman_p = spearmanr(
            temp[x_col],
            temp[y_col]
        )

        pearson_r, pearson_p = pearsonr(
            temp[x_col],
            temp[y_col]
        )

        return (
            n,
            spearman_r,
            spearman_p,
            pearson_r,
            pearson_p
        )

    else:

        return (
            n,
            np.nan,
            np.nan,
            np.nan,
            np.nan
        )


# ==========================================
# 12. Lagged correlation analysis
# ==========================================

print(
    f"\nRunning lagged correlation analysis..."
)

print(
    f"Independent variable: {INDEPENDENT_VARIABLE}"
)

print(
    f"Testing lag = 0 to {MAX_LAG}"
)


results = []

# 用于保存每一对关系实际匹配的数据
lagged_data_all = []


# ------------------------------------------
# 遍历 DS / ADS pair
# ------------------------------------------

for ds_name, ads_name in ds_ads_mapping.items():

    print(
        f"\nAnalyzing:"
        f" {ds_name} vs {ads_name}"
    )

    # --------------------------------------
    # 提取 DS
    # --------------------------------------

    ds_temp = ds_summary[
        ds_summary["DS"] == ds_name
    ][
        [
            "Environment",
            "Time_parsed",
            "DS_delta",
            "DS_rate",
            "DS_CLR_delta",
            "DS_CLR_rate"
        ]
    ].copy()


    # --------------------------------------
    # 提取 ADS
    # --------------------------------------

    ads_temp = ads_summary[
        ads_summary["ADS"] == ads_name
    ][
        [
            "Environment",
            "Time_parsed",
            "ADS_delta",
            "ADS_rate",
            "ADS_CLR_delta",
            "ADS_CLR_rate"
        ]
    ].copy()


    # --------------------------------------
    # 合并
    #
    # 先获得每个 Environment 内
    # DS 与 ADS 在同一时间点的数据
    # --------------------------------------

    merged = pd.merge(
        ds_temp,
        ads_temp,
        on=[
            "Environment",
            "Time_parsed"
        ],
        how="inner"
    )


    # 排序非常重要
    merged = merged.sort_values(
        [
            "Environment",
            "Time_parsed"
        ]
    ).reset_index(
        drop=True
    )


    # ======================================
    # 测试不同 lag
    # ======================================

    for lag in range(
        MAX_LAG + 1
    ):

        lagged = merged.copy()


        # ==================================
        # DS 作为领先变量
        #
        # DS(t)
        #      ↓
        # ADS(t + lag)
        # ==================================

        if INDEPENDENT_VARIABLE == "DS":

            lagged["X_delta"] = (
                lagged["DS_delta"]
            )

            lagged["Y_delta"] = (
                lagged
                .groupby("Environment")[
                    "ADS_delta"
                ]
                .shift(-lag)
            )


            lagged["X_rate"] = (
                lagged["DS_rate"]
            )

            lagged["Y_rate"] = (
                lagged
                .groupby("Environment")[
                    "ADS_rate"
                ]
                .shift(-lag)
            )


            lagged["X_CLR_delta"] = (
                lagged["DS_CLR_delta"]
            )

            lagged["Y_CLR_delta"] = (
                lagged
                .groupby("Environment")[
                    "ADS_CLR_delta"
                ]
                .shift(-lag)
            )


            lagged["X_CLR_rate"] = (
                lagged["DS_CLR_rate"]
            )

            lagged["Y_CLR_rate"] = (
                lagged
                .groupby("Environment")[
                    "ADS_CLR_rate"
                ]
                .shift(-lag)
            )


            # X 时间
            lagged["X_time"] = (
                lagged["Time_parsed"]
            )

            # Y 对应的未来时间
            lagged["Y_time"] = (
                lagged
                .groupby("Environment")[
                    "Time_parsed"
                ]
                .shift(-lag)
            )


        # ==================================
        # ADS 作为领先变量
        #
        # ADS(t)
        #      ↓
        # DS(t + lag)
        # ==================================

        elif INDEPENDENT_VARIABLE == "ADS":

            lagged["X_delta"] = (
                lagged["ADS_delta"]
            )

            lagged["Y_delta"] = (
                lagged
                .groupby("Environment")[
                    "DS_delta"
                ]
                .shift(-lag)
            )


            lagged["X_rate"] = (
                lagged["ADS_rate"]
            )

            lagged["Y_rate"] = (
                lagged
                .groupby("Environment")[
                    "DS_rate"
                ]
                .shift(-lag)
            )


            lagged["X_CLR_delta"] = (
                lagged["ADS_CLR_delta"]
            )

            lagged["Y_CLR_delta"] = (
                lagged
                .groupby("Environment")[
                    "DS_CLR_delta"
                ]
                .shift(-lag)
            )


            lagged["X_CLR_rate"] = (
                lagged["ADS_CLR_rate"]
            )

            lagged["Y_CLR_rate"] = (
                lagged
                .groupby("Environment")[
                    "DS_CLR_rate"
                ]
                .shift(-lag)
            )


            lagged["X_time"] = (
                lagged["Time_parsed"]
            )

            lagged["Y_time"] = (
                lagged
                .groupby("Environment")[
                    "Time_parsed"
                ]
                .shift(-lag)
            )


        else:

            raise ValueError(
                "INDEPENDENT_VARIABLE "
                "must be 'DS' or 'ADS'"
            )


        # ==================================
        # Calculate actual lag days
        # ==================================

        lagged["Lag_days"] = (
            lagged["Y_time"]
            -
            lagged["X_time"]
        ).dt.days


        # ==================================
        # Raw TPM difference correlation
        # ==================================

        (
            n_delta,
            spearman_r_delta,
            spearman_p_delta,
            pearson_r_delta,
            pearson_p_delta
        ) = calculate_correlation(
            lagged,
            "X_delta",
            "Y_delta"
        )


        # ==================================
        # CLR difference correlation
        # ==================================

        (
            n_clr_delta,
            spearman_r_clr,
            spearman_p_clr,
            pearson_r_clr,
            pearson_p_clr
        ) = calculate_correlation(
            lagged,
            "X_CLR_delta",
            "Y_CLR_delta"
        )


        # ==================================
        # Raw TPM rate correlation
        # ==================================

        (
            n_rate,
            spearman_r_rate,
            spearman_p_rate,
            pearson_r_rate,
            pearson_p_rate
        ) = calculate_correlation(
            lagged,
            "X_rate",
            "Y_rate"
        )


        # ==================================
        # CLR rate correlation
        # ==================================

        (
            n_clr_rate,
            spearman_r_clr_rate,
            spearman_p_clr_rate,
            pearson_r_clr_rate,
            pearson_p_clr_rate
        ) = calculate_correlation(
            lagged,
            "X_CLR_rate",
            "Y_CLR_rate"
        )


        # ==================================
        # Lag day statistics
        # ==================================

        valid_lag_days = lagged[
            "Lag_days"
        ].dropna()


        if len(valid_lag_days) > 0:

            lag_days_median = valid_lag_days.median()

            lag_days_mean = valid_lag_days.mean()

            lag_days_min = valid_lag_days.min()

            lag_days_max = valid_lag_days.max()

            lag_days_q25 = valid_lag_days.quantile(
                0.25
            )

            lag_days_q75 = valid_lag_days.quantile(
                0.75
            )

        else:

            lag_days_median = np.nan
            lag_days_mean = np.nan
            lag_days_min = np.nan
            lag_days_max = np.nan
            lag_days_q25 = np.nan
            lag_days_q75 = np.nan


        # ==================================
        # Save result
        # ==================================

        results.append({

            "Independent_variable":
                INDEPENDENT_VARIABLE,

            "DS":
                ds_name,

            "ADS":
                ads_name,

            "Lag":
                lag,

            # ------------------------------
            # Actual lag time
            # ------------------------------

            "Lag_days_median":
                lag_days_median,

            "Lag_days_mean":
                lag_days_mean,

            "Lag_days_min":
                lag_days_min,

            "Lag_days_Q25":
                lag_days_q25,

            "Lag_days_Q75":
                lag_days_q75,

            "Lag_days_max":
                lag_days_max,


            # ------------------------------
            # Raw TPM delta
            # ------------------------------

            "N_delta":
                n_delta,

            "Spearman_r_delta":
                spearman_r_delta,

            "Spearman_p_delta":
                spearman_p_delta,

            "Pearson_r_delta":
                pearson_r_delta,

            "Pearson_p_delta":
                pearson_p_delta,


            # ------------------------------
            # CLR delta
            # ------------------------------

            "N_CLR_delta":
                n_clr_delta,

            "Spearman_r_CLR_delta":
                spearman_r_clr,

            "Spearman_p_CLR_delta":
                spearman_p_clr,

            "Pearson_r_CLR_delta":
                pearson_r_clr,

            "Pearson_p_CLR_delta":
                pearson_p_clr,


            # ------------------------------
            # Raw TPM rate
            # ------------------------------

            "N_rate":
                n_rate,

            "Spearman_r_rate":
                spearman_r_rate,

            "Spearman_p_rate":
                spearman_p_rate,

            "Pearson_r_rate":
                pearson_r_rate,

            "Pearson_p_rate":
                pearson_p_rate,


            # ------------------------------
            # CLR rate
            # ------------------------------

            "N_CLR_rate":
                n_clr_rate,

            "Spearman_r_CLR_rate":
                spearman_r_clr_rate,

            "Spearman_p_CLR_rate":
                spearman_p_clr_rate,

            "Pearson_r_CLR_rate":
                pearson_r_clr_rate,

            "Pearson_p_CLR_rate":
                pearson_p_clr_rate
        })


        # ==================================
        # 保存具体 lag 数据
        # ==================================

        lagged_output = lagged.copy()

        lagged_output["DS"] = ds_name

        lagged_output["ADS"] = ads_name

        lagged_output["Lag"] = lag

        lagged_output["Independent_variable"] = (
            INDEPENDENT_VARIABLE
        )

        lagged_data_all.append(
            lagged_output
        )


# ==========================================
# 13. Save correlation results
# ==========================================

print(
    "\nSaving correlation results..."
)


results_df = pd.DataFrame(
    results
)


results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "DS_ADS_lagged_correlation_results.csv"
    ),
    index=False
)


# ==========================================
# Save matched lagged data
# ==========================================

lagged_data_df = pd.concat(
    lagged_data_all,
    ignore_index=True
)


lagged_data_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "DS_ADS_lagged_matched_data.csv"
    ),
    index=False
)


# ==========================================
# 14. Plot lag correlation curves
# ==========================================

print(
    "Generating lag correlation plots..."
)


n_pairs = len(
    ds_ads_mapping
)

cols = 4

rows = (
    n_pairs + cols - 1
) // cols


fig, axes = plt.subplots(
    rows,
    cols,
    figsize=(
        18,
        4 * rows
    )
)

axes = axes.flatten()


for idx, (
    ds_name,
    ads_name
) in enumerate(
    ds_ads_mapping.items()
):

    ax = axes[idx]

    subset = results_df[
        (
            results_df["DS"] == ds_name
        )
        &
        (
            results_df["ADS"] == ads_name
        )
    ].sort_values(
        "Lag"
    )


    ax.plot(
        subset["Lag"],
        subset["Spearman_r_CLR_delta"],
        marker="o",
        linewidth=1.5
    )


    ax.axhline(
        0,
        linestyle="--",
        linewidth=1,
        color="gray"
    )


    ax.set_xlabel(
        "Lag (sampling intervals)"
    )

    ax.set_ylabel(
        "Spearman r"
    )


    ax.set_title(
        f"{ds_name} vs {ads_name}"
    )


    # 标记显著结果

    for _, row in subset.iterrows():

        if (
            pd.notna(
                row["Spearman_p_CLR_delta"]
            )
            and
            row["Spearman_p_CLR_delta"] < 0.05
        ):

            ax.text(
                row["Lag"],
                row["Spearman_r_CLR_delta"],
                "*",
                fontsize=14,
                ha="center",
                va="bottom"
            )


# 隐藏多余子图

for i in range(
    idx + 1,
    len(axes)
):

    axes[i].axis(
        "off"
    )


plt.tight_layout()


plot_path = os.path.join(
    OUTPUT_DIR,
    "Lag_vs_Spearman_CLR_delta.pdf"
)


plt.savefig(
    plot_path,
    dpi=300,
    bbox_inches="tight"
)


plt.close()


# ==========================================
# Finished
# ==========================================

print(
    "\n========================================"
)

print(
    "Analysis finished!"
)

print(
    "========================================"
)

print(
    f"\nIndependent variable:"
    f" {INDEPENDENT_VARIABLE}"
)

print(
    f"\nLag tested:"
    f" 0 to {MAX_LAG}"
)

print(
    f"\nResults saved to:"
    f"\n{OUTPUT_DIR}"
)

Loading data...
Saving processed data...

Running lagged correlation analysis...
Independent variable: ADS
Testing lag = 0 to 3

Analyzing: Cas vs Anti_CRISPR

Analyzing: RM vs Anti_RM

Analyzing: CBASS vs Anti_CBASS

Analyzing: Pycsar vs Anti_Pycsar

Analyzing: Dnd vs Anti_Dnd

Analyzing: Gabija vs Anti_Gabija

Analyzing: Retron vs Anti_Retron

Analyzing: Thoeris vs Anti_Thoeris

Saving correlation results...
Generating lag correlation plots...

Analysis finished!

Independent variable: ADS

Lag tested: 0 to 3

Results saved to:
D:\softerware_scientific_research\python_all_inone\py learning\pythonProject\prophage_project\huiyuan_ADS_DS\DS_ADS_analysis_Lag_ADS
